In [2]:
## Retrieval augmented generation

import os
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
os.environ['GOOGLE_API_KEY']=os.getenv("GOOGLE_API_KEY")


In [4]:
from llama_index.llms.gemini import Gemini
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Initialize Gemini LLM (for text generation)
llm = Gemini(model="models/gemini-2.5-flash")

# Use local HuggingFace embedding model (no API calls, no rate limits)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

documents=SimpleDirectoryReader("data").load_data()


C:\Users\bibhu\AppData\Local\Temp\ipykernel_24880\1279903659.py:6: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm = Gemini(model="models/gemini-2.5-flash")
C:\Users\bibhu\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bibhu\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingf

In [5]:
documents

[Document(id_='cc50ef31-46ba-43ff-9dd5-7eb90fb902ab', embedding=None, metadata={'page_label': '1', 'file_name': 'exp3_Embedded.pdf', 'file_path': 'd:\\GenAI\\Basic Rag\\data\\exp3_Embedded.pdf', 'file_type': 'application/pdf', 'file_size': 2865850, 'creation_date': '2026-01-30', 'last_modified_date': '2026-01-27'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=None, image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='231ab497-d791-4898-9334-33444b804866', embedding=None, metadata={'page_label': '2', 'file_name': 'exp3_Embedded.pdf', 'file_path': 'd:\\GenAI\\Basic Rag\\data\\exp3_Embedded.pdf', 'file_type': 'applic

In [6]:
index=VectorStoreIndex.from_documents(documents,show_progress=True)

Generating embeddings: 100%|██████████| 8/8 [00:01<00:00,  6.26it/s]


In [7]:
index

In [8]:
query_engine=index.as_query_engine()

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import SimilarityPostprocessor

retriever=VectorIndexRetriever(index=index,similarity_top_k=4)
postprocessor=SimilarityPostprocessor(similarity_cutoff=0.80)

query_engine=RetrieverQueryEngine(retriever=retriever,
                                node_postprocessors=[postprocessor])


In [10]:
response=query_engine.query("What all skills are required for Scalecode.ai?")

In [14]:
from llama_index.core.response.pprint_utils import pprint_response
pprint_response(response,show_source=True)
print(response)


Final Response: Empty Response
Empty Response


In [ ]:
simple_query_engine = index.as_query_engine()
simple_response = simple_query_engine.query("What all skills are required for Scalecode.ai?")
print("Simple Query Response:")
print(simple_response)

Simple Query Response:
Salescode.ai requires candidates to possess proven expertise in backend or full-stack software development, with a preference for Python or Java. A deep understanding of system architecture, LLM workflows, and reasoning pipelines is essential, along with the ability to build the entire stack for AI systems.

Candidates should have experience with at least one end-to-end production-level project, ideally involving APIs, cloud deployment, or data pipelines. Key responsibilities include contributing to the development of Agentic AI systems (such as co-pilots, chatbots, and autonomous agents), building and maintaining backend components and APIs, and integrating tools like LlamaIndex, LangChain, and vector search systems into production applications.

The role also demands the ability to assist in deploying and optimizing AI agent pipelines using cloud environments like AWS or Azure. Strong coding skills are necessary, including writing clean, maintainable code, part

In [16]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.postprocessor import SimilarityPostprocessor

retriever_adjusted = VectorIndexRetriever(index=index, similarity_top_k=4)
postprocessor_adjusted = SimilarityPostprocessor(similarity_cutoff=0.5)  # Lower threshold

query_engine_adjusted = RetrieverQueryEngine(
    retriever=retriever_adjusted,
    node_postprocessors=[postprocessor_adjusted]
)

response_adjusted = query_engine_adjusted.query("What all skills are required for Scalecode.ai?")
print("\nAdjusted Query Response (0.5 cutoff):")
print(response_adjusted)


Adjusted Query Response (0.5 cutoff):
For a Software Engineer role at Salescode.ai, candidates should possess strong backend or full-stack software development expertise, with a preference for Python or Java. A deep understanding of system architecture, LLM workflows, and reasoning pipelines is essential, along with the ability to build the entire stack rather than just integrating existing tools.

Key responsibilities that imply required skills include:
*   Contributing to the development of Agentic AI systems, such as co-pilots, chatbots, and autonomous agents.
*   Building and maintaining backend components and APIs for real-time AI products.
*   Integrating tools like LlamaIndex, LangChain, and vector search systems into production applications.
*   Assisting in deploying and optimizing AI agent pipelines using cloud environments like AWS or Azure.
*   Collaborating with cross-functional teams.
*   Writing clean, maintainable code, participating in code reviews, testing, and docum

In [ ]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

query_engine = index.as_query_engine()
response = query_engine.query("What are transformers?")
print(response)
